In [4]:
import torch
import torchaudio
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
from PIL import Image
import re
import os
import glob
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, mean_absolute_error, mean_squared_error
import numpy as np
import warnings
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import time
from datetime import timedelta

# Set torchaudio backend to avoid torchcodec dependency
try:
    torchaudio.set_audio_backend("soundfile")
    print("✅ Using soundfile backend for torchaudio")
except Exception as e:
    print(f"⚠️ Could not set soundfile backend: {e}")
    print("   Trying with default backend...")

# Check GPU availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU device:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())
    print("Current GPU:", torch.cuda.current_device())
    print("GPU memory allocated:", torch.cuda.memory_allocated(0) / 1024**3, "GB")
    print("GPU memory reserved:", torch.cuda.memory_reserved(0) / 1024**3, "GB")
else:
    print("⚠️ WARNING: No GPU detected! Running on CPU will be extremely slow.")

# Disable tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Paths
image_path = r"D:\Dementia-Thesis - Don't access without permission\Cookie-Theft-stimulus.png"
audio_dir = r"D:\Dementia-Thesis - Don't access without permission\Audio_cookie"
ground_truth_path = r"D:\Dementia-Thesis - Don't access without permission\2_classes.csv"
output_csv_path = r"D:\Dementia-Thesis - Don't access without permission\qwenaudio_zero_shot.csv"
qwen_output_dir = r"D:\Dementia-Thesis - Don't access without permission\qwenaudio_outputs_zero_shot"

# Create output directory for Qwen outputs if it doesn't exist
os.makedirs(qwen_output_dir, exist_ok=True)

# Debug mode
DEBUG_MODE = False
DEBUG_SAMPLES = 5

print("\n" + "="*80)
print("🎯 ZERO-SHOT LEARNING MODE")
print("="*80)
print("ℹ️  This uses ZERO-SHOT learning:")
print("   - Model receives NO example audio analyses")
print("   - Model relies purely on its pre-trained knowledge")
print("   - Evaluates model's inherent understanding of dementia speech")
print("   - No training or fine-tuning required")
print("="*80 + "\n")

print("\n" + "="*80)
print("FREEING MEMORY BEFORE MODEL LOADING...")
print("="*80)

# Clean up memory
for var in ['qwen_model', 'qwen_processor']:
    if var in dir():
        exec(f"del {var}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB allocated")

print("✅ Memory cleanup complete!\n")

# Load ground truth
print("Loading ground truth data...")
df_gt = pd.read_csv(ground_truth_path)

if 'dx' in df_gt.columns:
    df_gt['dx'] = df_gt['dx'].apply(lambda x: str(x).lower() if pd.notna(x) else None)

print(f"Ground truth loaded: {len(df_gt)} total records")

df_gt_valid = df_gt[df_gt['dx'].notna()].copy()
print(f"Records with valid dx labels: {len(df_gt_valid)}")
print(f"Unique dx labels: {df_gt_valid['dx'].unique()}")

# Get all audio files
audio_extensions = ['*.wav', '*.mp3', '*.flac', '*.m4a', '*.ogg']
audio_files_all = []
for ext in audio_extensions:
    audio_files_all.extend(glob.glob(os.path.join(audio_dir, ext)))
audio_files_all = sorted(audio_files_all)

print(f"\nFound {len(audio_files_all)} total audio files")

# Extract IDs from audio filenames (remove PAR_ prefix if present)
def extract_id_from_filename(filepath):
    """Extract ID from filename, removing PAR_ prefix if present"""
    basename = os.path.splitext(os.path.basename(filepath))[0]
    # Remove PAR_ prefix if it exists
    if basename.startswith('PAR_'):
        return basename[4:]  # Remove first 4 characters 'PAR_'
    return basename

# Filter to only process files with valid dx labels
valid_ids = set(df_gt_valid['id'].astype(str))
audio_files = [f for f in audio_files_all if extract_id_from_filename(f) in valid_ids]

skipped_count = len(audio_files_all) - len(audio_files)
print(f"Files to process (with valid dx): {len(audio_files)}")
print(f"Files skipped (no dx label): {skipped_count}")

# Load Cookie Theft image
image = Image.open(image_path)

print("\n" + "="*80)
print("LOADING QWEN 2 AUDIO MODEL...")
print("="*80)

try:
    # Load Qwen 2 Audio - End-to-end audio understanding model
    model_name = "Qwen/Qwen2-Audio-7B-Instruct"
    
    print(f"Loading {model_name}...")
    print("ℹ️  Qwen 2 Audio directly analyzes speech including:")
    print("    - Pauses and silence duration")
    print("    - Filler words (uh, um, oh)")
    print("    - Speech rate and hesitations")
    print("    - Prosodic features (pitch, rhythm)")
    print("    - Plus linguistic content\n")
    
    qwen_processor = AutoProcessor.from_pretrained(model_name)
    qwen_model = Qwen2AudioForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    
    print(f"✅ Qwen 2 Audio model loaded successfully!")
    if torch.cuda.is_available():
        print(f"GPU memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
except Exception as e:
    print(f"❌ Qwen 2 Audio loading failed: {str(e)}")
    raise

print("="*80 + "\n")

# ============================================================================
# PROCESS AUDIO FILES
# ============================================================================

print("="*80)
print("STARTING AUDIO PROCESSING WITH ZERO-SHOT LEARNING...")
print("="*80)

# Initialize results storage
results = []
processing_count = 0
processing_diagnosis_counts = {'control': 0, 'probablead': 0, 'unknown': 0, 'error': 0}

# Timing variables
file_processing_times = []
start_time_overall = time.time()

# Process audio files
if DEBUG_MODE:
    files_to_process = audio_files[:DEBUG_SAMPLES]
    print(f"\n🔍 DEBUG MODE: Processing first {DEBUG_SAMPLES} files only\n")
else:
    files_to_process = audio_files

total_files = len(files_to_process)

for idx, audio_file in enumerate(files_to_process):
    file_start_time = time.time()
    file_name = os.path.basename(audio_file)
    
    # Extract ID (remove PAR_ prefix if present)
    file_id = extract_id_from_filename(audio_file)
    
    # Get ground truth for this file
    gt_row = df_gt_valid[df_gt_valid['id'].astype(str) == file_id]
    if len(gt_row) > 0:
        actual_dx = gt_row.iloc[0]['dx']
        actual_mmse = gt_row.iloc[0]['mmse'] if 'mmse' in gt_row.columns and pd.notna(gt_row.iloc[0]['mmse']) else None
    else:
        actual_dx = 'unknown'
        actual_mmse = None
    
    try:
        # Step 1: Analyze audio directly with Qwen 2 Audio using ZERO-SHOT LEARNING
        print(f"\n🎤 [{idx+1}/{total_files}] Analyzing audio: {file_name}") if DEBUG_MODE and processing_count < DEBUG_SAMPLES else None
        
        # Prepare ZERO-SHOT prompt for Qwen 2 Audio (NO EXAMPLES)
        conversation = [
            {
                "role": "system",
                "content": "You are an expert clinical neuropsychologist specializing in dementia assessment through speech analysis. You must be objective and base your diagnosis ONLY on the speech patterns you observe."
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": audio_file},
                    {"type": "text", "text": """Analyze this Cookie Theft picture description audio and classify into ONE of these TWO categories:

1. "probablead" - Probable Alzheimer's Disease
   INDICATORS: Frequent pauses, many filler words (um, uh), word-finding difficulties, incomplete descriptions, confused or tangential speech, slow speech rate, repetitive phrases, prosodic abnormalities

2. "control" - Healthy/Normal Cognition  
   INDICATORS: Fluent speech, minimal pauses, coherent narrative, rich descriptive detail, normal speech rate, organized thought flow, minimal fillers

CRITICAL: Assess objectively based on ACTUAL speech patterns observed. Both diagnoses are equally prevalent in this dataset.

Evaluate BOTH dimensions:

LINGUISTIC CONTENT:
- Description completeness and detail
- Logical coherence and organization  
- Word-finding difficulties
- Tangential or repetitive speech

ACOUSTIC/PROSODIC FEATURES:
- Pause frequency and duration (long/frequent pauses → impairment)
- Filler word frequency (uh, um, oh → impairment)
- Speech rate (slow/hesitant → impairment)
- Prosody abnormalities

Provide assessment in EXACTLY this format:

DIAGNOSIS: [probablead OR control]
MMSE_SCORE: [0-30]
REASONING: [Detailed analysis of BOTH linguistic and acoustic features observed]

MMSE Score Guidelines:
- 24-30: Normal → control
- 18-23: Mild impairment → control or probablead (depends on severity)
- 10-17: Moderate impairment → probablead
- 0-9: Severe impairment → probablead

Base your diagnosis on the actual speech patterns you hear in this recording."""}
                ]
            }
        ]
        
        # Process with Qwen 2 Audio
        text_prompt = qwen_processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=False
        )
        
        # Load and process audio
        try:
            waveform, sample_rate = torchaudio.load(audio_file, backend="soundfile")
        except:
            import soundfile as sf
            audio_array_raw, sample_rate = sf.read(audio_file)
            waveform = torch.from_numpy(audio_array_raw).float()
            if len(waveform.shape) == 1:
                waveform = waveform.unsqueeze(0)
            else:
                waveform = waveform.T
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        inputs = qwen_processor(
            text=text_prompt,
            audios=waveform.squeeze().numpy(),
            sampling_rate=sample_rate,
            return_tensors="pt",
            padding=True
        )
        inputs = {k: v.to(qwen_model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            generate_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                top_k=40,
                repetition_penalty=1.05
            )
        
        # Extract only the generated part (not the prompt)
        generated_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
        diagnosis_result = qwen_processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]
        
        # Clean up
        del inputs, generate_ids, generated_ids, waveform
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # PRINT QWEN OUTPUT
        print(f"\n{'='*80}")
        print(f"📄 FILE ID: {file_id} | Audio: {file_name}")
        print(f"{'='*80}")
        print(f"\n🧠 QWEN 2 AUDIO ANALYSIS (ZERO-SHOT):")
        print(f"{'-'*80}")
        print(diagnosis_result[:600])
        print(f"{'-'*80}\n")
        
        # Save Qwen output to file
        qwen_output_path = os.path.join(qwen_output_dir, f"{file_id}_qwen_zeroshot_analysis.txt")
        with open(qwen_output_path, 'w', encoding='utf-8') as f:
            f.write(f"File ID: {file_id}\n")
            f.write(f"Audio File: {file_name}\n")
            f.write(f"Learning Mode: ZERO-SHOT\n")
            f.write(f"{'='*80}\n")
            f.write(f"QWEN 2 AUDIO ANALYSIS:\n")
            f.write(f"{'='*80}\n")
            f.write(diagnosis_result)
        
        if DEBUG_MODE and processing_count < DEBUG_SAMPLES:
            print(f"\n{'='*80}")
            print(f"🔍 DEBUG OUTPUT #{processing_count + 1} - File: {file_id}")
            print(f"{'='*80}")
            print(f"\n{'─'*80}")
            print(f"FULL RAW MODEL OUTPUT:")
            print(f"{'─'*80}")
            print(diagnosis_result)
            print(f"{'='*80}\n")
        
        # Extract diagnosis with improved pattern matching
        diagnosis = None
        
        # Try primary pattern: DIAGNOSIS: control/probablead
        diagnosis_match = re.search(r'DIAGNOSIS:\s*["\']?(\w+)["\']?', diagnosis_result, re.IGNORECASE)
        if diagnosis_match:
            extracted = diagnosis_match.group(1).lower().strip()
            if 'probable' in extracted or extracted == 'probablead' or extracted == 'ad':
                diagnosis = 'probablead'
            elif extracted == 'control' or extracted == 'normal' or extracted == 'healthy':
                diagnosis = 'control'
        
        # Fallback 1: Look for diagnosis anywhere in first 200 chars
        if not diagnosis:
            first_part = diagnosis_result[:200].lower()
            # Count mentions of each diagnosis type
            control_mentions = len(re.findall(r'\bcontrol\b|\bnormal cognition\b|\bhealthy\b', first_part))
            ad_mentions = len(re.findall(r'\bprobablead\b|\bprobable ad\b|\balzheimer|\bdementia\b|\bcognitive impairment\b', first_part))
            
            if ad_mentions > control_mentions:
                diagnosis = 'probablead'
            elif control_mentions > ad_mentions:
                diagnosis = 'control'
        
        # Fallback 2: Look at MMSE score if we have it
        if not diagnosis:
            mmse_match_temp = re.search(r'MMSE[_\s]*SCORE[:\s]+(\d+)', diagnosis_result, re.IGNORECASE)
            if mmse_match_temp:
                temp_score = int(mmse_match_temp.group(1))
                if temp_score >= 24:
                    diagnosis = 'control'
                elif temp_score < 24:
                    diagnosis = 'probablead'
        
        # Final fallback: Look for any mention in entire response
        if not diagnosis:
            if re.search(r'\bprobable\s*ad\b|\bprobablead\b|\balzheimer|\bdementia\b|\bimpairment\b', diagnosis_result, re.IGNORECASE):
                diagnosis = 'probablead'
            elif re.search(r'\bcontrol\b|\bhealthy\b|\bnormal\b', diagnosis_result, re.IGNORECASE):
                diagnosis = 'control'
            else:
                diagnosis = 'unknown'
        
        # Extract MMSE
        mmse_score = -1
        mmse_match = re.search(r'MMSE[_\s]*SCORE[:\s]+([^\n]+)', diagnosis_result, re.IGNORECASE)
        if mmse_match:
            mmse_text = mmse_match.group(1).strip()
            if not any(x in mmse_text.lower() for x in ["n/a", "not applicable", "cannot", "unable"]):
                nums = re.findall(r'\b(\d+)\b', mmse_text)
                for num_str in nums:
                    num = int(num_str)
                    if 0 <= num <= 30:
                        mmse_score = num
                        break
        
        # Extract reasoning
        reasoning_match = re.search(r'REASONING:(.+?)(?:\n\n|\Z)', diagnosis_result, re.IGNORECASE | re.DOTALL)
        reasoning = reasoning_match.group(1).strip() if reasoning_match else diagnosis_result[:200]
        
        # Calculate file processing time
        file_end_time = time.time()
        file_processing_time = file_end_time - file_start_time
        file_processing_times.append(file_processing_time)
        
        # Calculate timing statistics
        avg_time_per_file = np.mean(file_processing_times)
        remaining_files = total_files - (idx + 1)
        estimated_time_remaining = avg_time_per_file * remaining_files
        
        processing_diagnosis_counts[diagnosis] += 1
        processing_count += 1
        
        results.append({
            'id': file_id,
            'file': file_name,
            'transcription': '[Qwen 2 Audio - Direct Analysis - Zero-Shot]',
            'predicted_diagnosis': diagnosis,
            'predicted_mmse': mmse_score,
            'actual_diagnosis': actual_dx,
            'actual_mmse': actual_mmse if actual_mmse is not None else -1,
            'reasoning': reasoning,
            'audio_duration_sec': 0,
            'raw_model_output': diagnosis_result if DEBUG_MODE else '',
            'processing_time_sec': file_processing_time
        })
        
        # Print progress with timing AND ground truth comparison
        elapsed_time = time.time() - start_time_overall
        dx_match = "✅" if diagnosis == actual_dx else "❌"
        mmse_match = "✅" if mmse_score == actual_mmse else "❌" if actual_mmse is not None else "⚠️"
        
        print(f"✅ [{idx+1}/{total_files}] {file_id}")
        print(f"   Predicted: {diagnosis} (MMSE: {mmse_score if mmse_score != -1 else 'N/A'})")
        print(f"   Actual:    {actual_dx} (MMSE: {actual_mmse if actual_mmse is not None else 'N/A'})")
        print(f"   Match: DX {dx_match} | MMSE {mmse_match}")
        print(f"   Time: File {file_processing_time:.1f}s | Avg {avg_time_per_file:.1f}s | ETA {str(timedelta(seconds=int(estimated_time_remaining)))} | Elapsed {str(timedelta(seconds=int(elapsed_time)))}")
        
        # Progress update
        if processing_count % 10 == 0:
            print(f"\n📊 Running distribution after {processing_count} files:")
            for dx, count in processing_diagnosis_counts.items():
                pct = (count / processing_count * 100) if processing_count > 0 else 0
                print(f"  {dx}: {count} ({pct:.1f}%)")
            print()
        
    except Exception as e:
        print(f"❌ ERROR processing {file_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        
        file_end_time = time.time()
        file_processing_time = file_end_time - file_start_time
        
        results.append({
            'id': file_id,
            'file': file_name,
            'transcription': '[Error]',
            'predicted_diagnosis': 'error',
            'predicted_mmse': -1,
            'actual_diagnosis': actual_dx,
            'actual_mmse': actual_mmse if actual_mmse is not None else -1,
            'reasoning': f'Processing error: {str(e)[:100]}',
            'audio_duration_sec': 0,
            'raw_model_output': '',
            'processing_time_sec': file_processing_time
        })
        processing_diagnosis_counts['error'] += 1
        continue

# Final timing summary
total_time = time.time() - start_time_overall
print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE!")
print(f"Total files processed: {len(results)}")
print(f"Total time: {str(timedelta(seconds=int(total_time)))}")
print(f"Average time per file: {np.mean(file_processing_times):.2f} seconds")
print(f"Min time: {np.min(file_processing_times):.2f}s | Max time: {np.max(file_processing_times):.2f}s")
print(f"Qwen 2 Audio outputs saved to: {qwen_output_dir}")
print('='*80)

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️ Could not set soundfile backend: module 'torchaudio' has no attribute 'set_audio_backend'
   Trying with default backend...
PyTorch version: 2.11.0.dev20251223+cu128
CUDA available: True
CUDA version: 12.8
GPU device: NVIDIA GeForce RTX 5090
Number of GPUs: 1
Current GPU: 0
GPU memory allocated: 0.0 GB
GPU memory reserved: 0.0 GB

🎯 ZERO-SHOT LEARNING MODE
ℹ️  This uses ZERO-SHOT learning:
   - Model receives NO example audio analyses
   - Model relies purely on its pre-trained knowledge
   - Evaluates model's inherent understanding of dementia speech
   - No training or fine-tuning required


FREEING MEMORY BEFORE MODEL LOADING...
GPU memory after cleanup: 0.00 GB allocated
✅ Memory cleanup complete!

Loading ground truth data...
Ground truth loaded: 498 total records
Records with valid dx labels: 498
Unique dx labels: ['probablead' 'control']

Found 552 total audio files
Files to process (with valid dx): 498
Files skipped (no dx label): 54

LOADING QWEN 2 AUDIO MODEL...
Loading Qw

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:06<00:00,  1.26s/it]
Keyword argument `audios` is not a valid argument for this processor and will be ignored.


✅ Qwen 2 Audio model loaded successfully!
GPU memory: 15.64 GB

STARTING AUDIO PROCESSING WITH ZERO-SHOT LEARNING...

📄 FILE ID: 001-0 | Audio: PAR_001-0.wav

🧠 QWEN 2 AUDIO ANALYSIS (ZERO-SHOT):
--------------------------------------------------------------------------------
DIAGNOSIS: control MMSE SCORE: 25 REASONING: The speaker has a fluent and coherent narrative with minimal pauses and fillers. The speech is detailed and organized, with rich descriptive detail. The speaker does not have any difficulty finding words or have tangential or repetitive speech. There are no acoustic or prosody abnormalities observed. Based on these features, the speaker has a normal level of cognition and is classified as control.
--------------------------------------------------------------------------------

✅ [1/498] 001-0
   Predicted: control (MMSE: 25)
   Actual:    probablead (MMSE: 18.0)
   Match: DX ❌ | MMSE ❌
   Time: File 3.3s | Avg 3.3s | ETA 0:27:06 | Elapsed 0:00:03

📄 FILE ID: 001-2 | Au

In [5]:
# ============================================================================
# CALCULATE METRICS AND SAVE RESULTS TO CSV
# ============================================================================

# Create DataFrame from results
df_results = pd.DataFrame(results)

# Count error/unknown predictions
error_count = (df_results['predicted_diagnosis'] == 'error').sum()
unknown_count = (df_results['predicted_diagnosis'] == 'unknown').sum()
valid_count = len(df_results) - error_count - unknown_count

print(f"\n{'='*80}")
print("RESULTS SUMMARY")
print('='*80)
print(f"\nTotal predictions: {len(df_results)}")
print(f"  Valid predictions: {valid_count}")
print(f"  Unknown predictions: {unknown_count}")
print(f"  Error predictions: {error_count}")

# Print diagnosis distribution (valid only)
valid_results = df_results[~df_results['predicted_diagnosis'].isin(['error', 'unknown'])]
if len(valid_results) > 0:
    print(f"\nDiagnosis distribution (valid predictions):")
    diagnosis_counts = valid_results['predicted_diagnosis'].value_counts()
    for dx, count in diagnosis_counts.items():
        pct = (count / len(valid_results) * 100)
        print(f"  {dx}: {count} ({pct:.1f}%)")

# MMSE statistics
valid_mmse = df_results[df_results['predicted_mmse'] >= 0]['predicted_mmse']
if len(valid_mmse) > 0:
    print(f"\nMMSE score statistics (valid predictions):")
    print(f"  Mean:   {valid_mmse.mean():.2f}")
    print(f"  Median: {valid_mmse.median():.2f}")
    print(f"  Std:    {valid_mmse.std():.2f}")
    print(f"  Range:  {valid_mmse.min()} - {valid_mmse.max()}")

# Audio duration statistics
if 'audio_duration_sec' in df_results.columns:
    valid_duration = df_results[df_results['audio_duration_sec'] > 0]['audio_duration_sec']
    if len(valid_duration) > 0:
        print(f"\nAudio duration statistics:")
        print(f"  Mean:   {valid_duration.mean():.2f} seconds")
        print(f"  Median: {valid_duration.median():.2f} seconds")
        print(f"  Range:  {valid_duration.min():.2f} - {valid_duration.max():.2f} seconds")

# ============================================================================
# MERGE WITH GROUND TRUTH AND CALCULATE METRICS
# ============================================================================

print(f"\n{'='*80}")
print("CALCULATING METRICS")
print('='*80)

# Merge predictions with ground truth
df_merged = df_results.merge(df_gt_valid, on='id', how='inner')
print(f"\nMerged {len(df_merged)} records with ground truth")

if len(df_merged) > 0:
    # Classification metrics (error/unknown count as wrong)
    y_true = df_merged['dx'].values
    y_pred = df_merged['predicted_diagnosis'].values
    
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(df_merged)
    accuracy_all = correct_predictions / total_predictions
    
    error_unknown_count = df_merged['predicted_diagnosis'].isin(['error', 'unknown']).sum()
    
    print(f"\n--- CLASSIFICATION METRICS (error/unknown count as WRONG) ---")
    print(f"Total with ground truth: {total_predictions}")
    print(f"Correct predictions: {correct_predictions}")
    print(f"Wrong predictions: {total_predictions - correct_predictions}")
    print(f"  - of which error/unknown: {error_unknown_count}")
    print(f"Accuracy (ALL): {accuracy_all:.4f}")
    
    # Calculate standard metrics only on valid predictions
    df_valid = df_merged[~df_merged['predicted_diagnosis'].isin(['error', 'unknown'])].copy()
    
    if len(df_valid) > 0:
        y_true_valid = df_valid['dx'].values
        y_pred_valid = df_valid['predicted_diagnosis'].values
        
        accuracy_valid = accuracy_score(y_true_valid, y_pred_valid)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true_valid, y_pred_valid, 
            average='binary', 
            pos_label='probablead',
            zero_division=0
        )
        
        # Confusion Matrix
        cm = confusion_matrix(y_true_valid, y_pred_valid, labels=['probablead', 'control'])
        tp, fn, fp, tn = cm.ravel()
        
        # Specificity and Sensitivity
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        print(f"\n--- METRICS ON VALID PREDICTIONS ONLY ---")
        print(f"Valid predictions evaluated: {len(df_valid)}")
        print(f"Accuracy:    {accuracy_valid:.4f}")
        print(f"Precision:   {precision:.4f}")
        print(f"Recall:      {recall:.4f}")
        print(f"F1-Score:    {f1:.4f}")
        print(f"Sensitivity: {sensitivity:.4f}")
        print(f"Specificity: {specificity:.4f}")
        
        print("\nConfusion Matrix (valid predictions):")
        print(f"                  Predicted")
        print(f"Actual       probableAD  control")
        print(f"probableAD     {tp:5d}    {fn:5d}")
        print(f"control        {fp:5d}    {tn:5d}")
        
        # Save confusion matrix visualization
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=['probableAD', 'control'],
                    yticklabels=['probableAD', 'control'],
                    cbar_kws={'label': 'Count'})
        plt.title('Confusion Matrix - Zero-Shot Learning (Qwen Audio)', fontsize=16, fontweight='bold')
        plt.ylabel('Actual Diagnosis', fontsize=12)
        plt.xlabel('Predicted Diagnosis', fontsize=12)
        
        # Add metrics text
        metrics_text = f'Accuracy: {accuracy_valid:.3f}\nPrecision: {precision:.3f}\nRecall: {recall:.3f}\nF1-Score: {f1:.3f}\nSensitivity: {sensitivity:.3f}\nSpecificity: {specificity:.3f}'
        plt.text(2.5, 0.5, metrics_text, fontsize=10, 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                verticalalignment='center')
        
        plt.tight_layout()
        
        # Save with same naming pattern as CSV
        cm_image_path = output_csv_path.replace('.csv', '_confusion_matrix.png')
        plt.savefig(cm_image_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"\n✅ Confusion matrix saved to: {cm_image_path}")
    else:
        accuracy_valid = precision = recall = f1 = sensitivity = specificity = 0
        tp = tn = fp = fn = 0
    
    # MMSE Metrics
    records_with_mmse = df_merged[df_merged['mmse'].notna()].copy()
    
    if len(records_with_mmse) > 0:
        correct_mmse = (records_with_mmse['mmse'] == records_with_mmse['predicted_mmse']).sum()
        total_mmse = len(records_with_mmse)
        error_mmse = (records_with_mmse['predicted_mmse'] == -1).sum()
        
        print(f"\n--- MMSE METRICS (error/-1 counts as WRONG) ---")
        print(f"Total with ground truth MMSE: {total_mmse}")
        print(f"Exact matches: {correct_mmse}")
        print(f"Wrong predictions: {total_mmse - correct_mmse}")
        print(f"  - of which error/unknown (-1): {error_mmse}")
        
        # MAE/RMSE only on valid predictions
        df_valid_mmse = records_with_mmse[records_with_mmse['predicted_mmse'] >= 0]
        
        if len(df_valid_mmse) > 0:
            y_true_mmse = df_valid_mmse['mmse'].values
            y_pred_mmse = df_valid_mmse['predicted_mmse'].values
            
            mae = mean_absolute_error(y_true_mmse, y_pred_mmse)
            mse = mean_squared_error(y_true_mmse, y_pred_mmse)
            rmse = np.sqrt(mse)
            
            print(f"\nRegression metrics (valid predictions, excluding -1):")
            print(f"MAE (Mean Absolute Error): {mae:.4f}")
            print(f"MSE (Mean Squared Error):  {mse:.4f}")
            print(f"RMSE (Root MSE):           {rmse:.4f}")
            print(f"Samples evaluated:         {len(df_valid_mmse)}")
        else:
            mae = mse = rmse = np.nan
    else:
        mae = mse = rmse = np.nan
        correct_mmse = total_mmse = error_mmse = 0
    
    # Create metrics summary rows
    metrics_summary = [
        {'id': '', 'file': '--- METRICS SUMMARY ---', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'METRIC', 'file': 'Accuracy (ALL, error=wrong)', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': accuracy_all, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'METRIC', 'file': 'Accuracy (valid only)', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': accuracy_valid, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'METRIC', 'file': 'Precision', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': precision, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'METRIC', 'file': 'Recall', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': recall, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'METRIC', 'file': 'F1-Score', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': f1, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'METRIC', 'file': 'Sensitivity', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': sensitivity, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'METRIC', 'file': 'Specificity', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': specificity, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': '', 'file': '', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'CONFUSION', 'file': 'True Positives', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': int(tp), 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'CONFUSION', 'file': 'True Negatives', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': int(tn), 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'CONFUSION', 'file': 'False Positives', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': int(fp), 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'CONFUSION', 'file': 'False Negatives', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': int(fn), 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': '', 'file': '', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'MMSE', 'file': 'MAE', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': mae, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'MMSE', 'file': 'MSE', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': mse, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'MMSE', 'file': 'RMSE', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': rmse, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': '', 'file': '', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': np.nan, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'COUNT', 'file': 'Total Predictions', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': len(df_results), 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'COUNT', 'file': 'Valid DX Predictions', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': len(df_valid) if len(df_valid) > 0 else 0, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'COUNT', 'file': 'Error/Unknown DX', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': error_unknown_count, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'COUNT', 'file': 'Valid MMSE Predictions', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': len(df_valid_mmse) if len(records_with_mmse) > 0 and len(df_valid_mmse) > 0 else 0, 'reasoning': '', 'audio_duration_sec': np.nan},
        {'id': 'COUNT', 'file': 'Error/Unknown MMSE (-1)', 'transcription': '', 'predicted_diagnosis': '', 'predicted_mmse': error_mmse, 'reasoning': '', 'audio_duration_sec': np.nan},
    ]
    
    df_with_metrics = pd.concat([df_results, pd.DataFrame(metrics_summary)], ignore_index=True)
else:
    df_with_metrics = df_results

# ============================================================================
# SAVE RESULTS TO CSV
# ============================================================================

try:
    df_with_metrics.to_csv(output_csv_path, index=False)
    print(f"\n{'='*80}")
    print(f"✅ RESULTS SAVED TO CSV!")
    print(f"{'='*80}")
    print(f"Output file: {output_csv_path}")
    print(f"Total records saved: {len(df_with_metrics)}")
    print(f"  - Predictions: {len(df_results)}")
    print(f"  - Metrics summary: {len(metrics_summary) if len(df_merged) > 0 else 0}")
    print(f"\nNote: {skipped_count} files were skipped (no dx label in ground truth)")
    print('='*80)
except Exception as e:
    print(f"\n❌ ERROR saving CSV: {str(e)}")
    print("Results are still available in df_results DataFrame")

print("\n✅ Done!")
print("\nZero-shot learning complete without examples.")


RESULTS SUMMARY

Total predictions: 498
  Valid predictions: 498
  Unknown predictions: 0
  Error predictions: 0

Diagnosis distribution (valid predictions):
  control: 347 (69.7%)
  probablead: 151 (30.3%)

MMSE score statistics (valid predictions):
  Mean:   24.03
  Median: 28.00
  Std:    6.29
  Range:  2 - 30

CALCULATING METRICS

Merged 498 records with ground truth

--- CLASSIFICATION METRICS (error/unknown count as WRONG) ---
Total with ground truth: 498
Correct predictions: 258
Wrong predictions: 240
  - of which error/unknown: 0
Accuracy (ALL): 0.5181

--- METRICS ON VALID PREDICTIONS ONLY ---
Valid predictions evaluated: 498
Accuracy:    0.5181
Precision:   0.5497
Recall:      0.3255
F1-Score:    0.4089
Sensitivity: 0.3255
Specificity: 0.7202

Confusion Matrix (valid predictions):
                  Predicted
Actual       probableAD  control
probableAD        83      172
control           68      175

✅ Confusion matrix saved to: D:\Dementia-Thesis - Don't access without perm